In [579]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from normalize_models import normalize_model, discover_manufacturer_from_model

import importlib
import normalize_models

importlib.reload(normalize_models);

In [580]:
df = pd.read_csv("data/vehicles.csv")

In [581]:
df.shape

(426880, 18)

In [582]:
original_rows = 426_880
print(original_rows)

426880


In [583]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   region        426880 non-null  object 
 2   price         426880 non-null  int64  
 3   year          425675 non-null  float64
 4   manufacturer  409234 non-null  object 
 5   model         421603 non-null  object 
 6   condition     252776 non-null  object 
 7   cylinders     249202 non-null  object 
 8   fuel          423867 non-null  object 
 9   odometer      422480 non-null  float64
 10  title_status  418638 non-null  object 
 11  transmission  424324 non-null  object 
 12  VIN           265838 non-null  object 
 13  drive         296313 non-null  object 
 14  size          120519 non-null  object 
 15  type          334022 non-null  object 
 16  paint_color   296677 non-null  object 
 17  state         426880 non-null  object 
dtypes: f

In [584]:
df.isna().sum()

id                   0
region               0
price                0
year              1205
manufacturer     17646
model             5277
condition       174104
cylinders       177678
fuel              3013
odometer          4400
title_status      8242
transmission      2556
VIN             161042
drive           130567
size            306361
type             92858
paint_color     130203
state                0
dtype: int64

Set the **price** range boundaries.

In [585]:
df = df[df["price"].between(1000, 250_000)]
df.shape

(380470, 18)

In [586]:
df["price"].agg(["min", "max", "mean", "std"])

min       1000.000000
max     249999.000000
mean     19535.253794
std      15050.231510
Name: price, dtype: float64

In [587]:
price_reduced_rows = df.shape[0]
removed_rows = original_rows - price_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by price filter: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by price filter: 46,410
Percentage removed: 10.9%


Set the **year** range boundaries.

In [588]:
df = df[df["year"].between(1995.0, 2026.0)]
df.shape

(364342, 18)

In [589]:
df["year"].agg(["min", "max", "mean", "std"])

min     1995.000000
max     2022.000000
mean    2012.453489
std        5.515889
Name: year, dtype: float64

In [590]:
year_reduced_rows = df.shape[0]
removed_rows = price_reduced_rows - year_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by year filter: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by year filter: 16,128
Percentage removed: 3.8%


Set the **odometer** range boundaries.

In [591]:
df = df[df["odometer"].between(1000.0, 250_000.0)]
df.shape

(350543, 18)

In [592]:
df["odometer"].agg(["min", "max", "mean", "std"])
df.shape

(350543, 18)

In [593]:
odometer_reduced_rows = df.shape[0]
removed_rows = year_reduced_rows - odometer_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by odometer filter: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by odometer filter: 13,799
Percentage removed: 3.2%


Drop unusable **model** rows.

In [594]:
df = df.dropna(subset=["model"])
df.shape

(347595, 18)

In [595]:
model_nan_reduced_rows = df.shape[0]
removed_rows = odometer_reduced_rows - model_nan_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by missing model filter: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by missing model filter: 2,948
Percentage removed: 0.7%


Drop rows that are not cars.

In [596]:
df = df[df["manufacturer"] != "harley-davidson"]

df = df[df["type"].str.lower() != "bus"]

non_car_manufacturers = "freightliner|international|hino|peterbilt|peterbuilt|yamaha|kenworth|workhorse|mack|kawasaki|polaris|thomas|sterling|winnebago"
non_car_manufacturers_mask =  df["model"].str.contains(non_car_manufacturers, case=False, na=False)
df = df[~non_car_manufacturers_mask]
df.shape

(345522, 18)

In [597]:
non_car_reduced_rows = df.shape[0]
removed_rows = model_nan_reduced_rows - non_car_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by non-car filter: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by non-car filter: 2,073
Percentage removed: 0.5%


Drop cars (rows) that are for salvage and only keep cars that have clean title.

In [598]:
df = df[(df["title_status"] == "clean") & (df["condition"] != "salvage")]
df.shape

(328411, 18)

In [599]:
condition_title_reduced_rows = df.shape[0]
removed_rows = non_car_reduced_rows - condition_title_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by condition and title filter: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by condition and title filter: 17,111
Percentage removed: 4.0%


Rename the (raw) manufacturer and model columns.

In [600]:
df = df.rename(
    columns={
        "manufacturer": "manufacturer_raw",
        "model": "model_raw",
    }
)


Discover missing manufacturers from the raw model strings.

In [601]:
df["make"] = df.apply(
    lambda row: discover_manufacturer_from_model(row["manufacturer_raw"], row["model_raw"]),
    axis=1,
)

Normalize the raw model strings.

In [602]:
df["model"] = df.apply(
    lambda row: normalize_model(row["make"], row["model_raw"]),
    axis=1,
)

Drop unusable makes and models

In [603]:
df = df[(df["make"] != "unknown") & (df["model"] != "unknown")]
df.shape


(326408, 20)

In [604]:
make_model_discovery_reduced_rows = df.shape[0]
removed_rows = condition_title_reduced_rows - make_model_discovery_reduced_rows
removed_pct = removed_rows / original_rows * 100
print(f"Rows removed by make and model discovery filters: {removed_rows:,}")
print(f"Percentage removed: {removed_pct:.1f}%")

Rows removed by make and model discovery filters: 2,003
Percentage removed: 0.5%


In [605]:
df.isna().sum()

id                       0
region                   0
price                    0
year                     0
manufacturer_raw      7084
model_raw                0
condition           125234
cylinders           137527
fuel                  1480
odometer                 0
title_status             0
transmission          1321
VIN                 112698
drive                95802
size                237110
type                 64008
paint_color          87352
state                    0
make                     0
model                    0
dtype: int64

Consolidate low (make,model) group counts.

In [606]:
PAIR_MIN = 50
OTHER_MIN = 50
MAKE_MIN = 50

# 1. Bucket rare original make/model pairs into make/other
pair_counts = df.groupby(["make", "model"])["model"].transform("size")
df.loc[pair_counts < PAIR_MIN, "model"] = "other"

# 2. Check make/other groups specifically
make_other_counts = df.groupby(["make", "model"])["model"].transform("size")

low_count_make_other = (
    (df["model"] == "other") &
    (make_other_counts < OTHER_MIN)
)

df.loc[low_count_make_other, "make"] = "other"
df.loc[low_count_make_other, "model"] = "other"

# 3. Also collapse makes that are globally rare
make_counts = df.groupby("make")["make"].transform("size")

rare_makes = make_counts < MAKE_MIN

df.loc[rare_makes, "make"] = "other"
df.loc[rare_makes, "model"] = "other"

Impute features that have **isna()** values.

In [607]:
def impute_feature(feature_column: str) -> None:
    df[feature_column] = df.groupby(["make", "model"])[feature_column].transform(
        lambda values: values.fillna(values.mode().iloc[0]) if not values.mode().empty else values
    )
    df[feature_column] = df.groupby('make')[feature_column].transform(
        lambda values:values.fillna(values.mode().iloc[0]) if not values.mode().empty else values
    )

In [608]:
impute_feature("condition")
impute_feature("cylinders")
impute_feature("fuel")
impute_feature("transmission")
impute_feature("drive")
impute_feature("size")
impute_feature("type")

Drop unnecessary features

In [609]:
df.drop(columns=['manufacturer_raw','model_raw','VIN','paint_color','title_status'],inplace=True)

In [610]:
df.isna().sum()

id              0
region          0
price           0
year            0
condition       0
cylinders       0
fuel            0
odometer        0
transmission    0
drive           0
size            0
type            0
state           0
make            0
model           0
dtype: int64

Introduce the car age feature.

In [611]:
df["age"] = 2026 - df["year"]

In [612]:
df.shape

(326408, 16)

In [613]:
df.to_csv("data/cars.csv", index=False)
